# 18 — A2A concepts: a standard boundary between agents

The Agent2Agent (A2A) protocol is the peer-level complement to MCP. MCP standardizes how an agent reaches *capabilities* (tools, resources, prompts); A2A standardizes how an agent works with *another agent* — an opaque peer with its own model, framework, and policies. Announced by Google in April 2025, donated to the Linux Foundation two months later, and at v1.0 with 150+ member organizations, it is the second pillar of the current interop stack.

**Discovery.** A remote agent publishes an *Agent Card* at `/.well-known/agent-card.json`: identity, skills, supported interfaces (transport bindings), and capabilities. v1.0 added JWS-signed cards so a client can verify who issued one.

**Tasks, not calls.** Where a tool call returns a value, delegation creates a *task* with an observable lifecycle: `SUBMITTED → WORKING → { INPUT_REQUIRED | COMPLETED | FAILED | CANCELED | REJECTED | AUTH_REQUIRED }`. Outputs arrive as *artifacts*; the `INPUT_REQUIRED` state makes multi-turn clarification part of the contract instead of an ad-hoc convention.

**Wire format.** Operations (`SendMessage`, `GetTask`, `CancelTask`, streaming and push variants) bind to JSON-RPC over HTTP(S), gRPC, or plain HTTP+JSON. Task states serialize as `TASK_STATE_*` strings.

**Lineage note.** IBM/BeeAI's REST-based *Agent Communication Protocol* (ACP) merged into A2A under the Linux Foundation in August 2025 — you will still meet the acronym in older material, but new work targets A2A.

**Where this fits the lab.** Notebook 04's supervisor–worker and handoff patterns lived inside one process. A2A is what those patterns look like across an organizational or vendor boundary. The cells below use the deterministic double in `lab_core.a2a_toy`; notebook 19 repeats the exercise with the official SDK over real HTTP.

Primary sources: [specification](https://a2a-protocol.org/latest/specification/), [Linux Foundation launch](https://www.linuxfoundation.org/press/linux-foundation-launches-the-agent2agent-protocol-project-to-enable-secure-intelligent-communication-between-ai-agents), [one-year adoption milestone](https://www.linuxfoundation.org/press/a2a-protocol-surpasses-150-organizations-lands-in-major-cloud-platforms-and-sees-enterprise-production-use-in-first-year), and the [ACP merger announcement](https://lfaidata.foundation/communityblog/2025/08/29/acp-joins-forces-with-a2a-under-the-linux-foundations-lf-ai-data/).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.a2a_toy import ToyAgentCard, ToyA2AServer, ToyA2ANetwork, ToyA2AClient

def forecast(text):
    forecasts = {'oslo': 'Snow, -3C', 'lisbon': 'Sunny, 21C'}
    city = next((c for c in forecasts if c in text.lower()), None)
    return {'artifact': forecasts[city] if city else 'no forecast available'}

def trip_desk(text):
    for city in ('Oslo', 'Lisbon'):
        if city.lower() in text.lower():
            return {'artifact': f'Itinerary for {city}: harbor walk, museum pass, one cafe.'}
    return {'input_required': 'Which city - Oslo or Lisbon?'}

network = ToyA2ANetwork()
network.register(ToyA2AServer(ToyAgentCard(
    'forecast-agent', 'Deterministic weather peer.', 'https://forecast.lab',
    skills=[{'id': 'get_forecast', 'name': 'Get forecast'}]), forecast))
network.register(ToyA2AServer(ToyAgentCard(
    'trip-agent', 'Plans a one-day itinerary.', 'https://trips.lab',
    skills=[{'id': 'plan_trip', 'name': 'Plan trip'}]), trip_desk))

client = ToyA2AClient(network)
print(json.dumps(client.discover('https://forecast.lab'), indent=2))  # the Agent Card

In [ ]:
# Immediate completion: delegate, get a terminal task back with an artifact.
task = client.send_message('https://forecast.lab', 'What is the weather in Oslo?')
print(task['status']['state'], '->', task['artifacts'][0]['parts'][0]['text'])

# INPUT_REQUIRED: the peer pauses the task and asks a question; the follow-up
# message carries the taskId, so the same task resumes and completes.
task = client.send_message('https://trips.lab', 'Plan me a day trip')
print(task['id'], task['status']['state'])
print('peer asks:', task['status']['update']['parts'][0]['text'])
task = client.send_message('https://trips.lab', 'Oslo please', task_id=task['id'])
print(task['status']['state'], '->', task['artifacts'][0]['parts'][0]['text'])

# Terminal tasks refuse cancellation with a protocol-level error.
try:
    client.cancel_task('https://trips.lab', task['id'])
except RuntimeError as exc:
    print('cancel refused:', exc)

print(json.dumps(client.transcript[-2], indent=2))  # one full JSON-RPC exchange

In [ ]:
# Notebook 04's supervisor-worker, across a wire: the supervisor stays local
# deterministic code; the workers are remote peers reached only through A2A.
# The toy network is single-threaded to keep traces deterministic - with a real
# transport you would fan out concurrently exactly as in notebook 02.
def ask_forecast(city):
    session = ToyA2AClient(network)  # one client session per delegation
    task = session.send_message('https://forecast.lab', f'weather in {city}')
    return f"{city}: {task['artifacts'][0]['parts'][0]['text']}"

print(' | '.join(ask_forecast(city) for city in ('Oslo', 'Lisbon')))
print('wire messages in the main session:', len(client.transcript))

**Trade-off.** A2A buys vendor-neutral delegation at the cost of a network hop, an auth story, and a bigger trust decision than MCP: Agent Cards, status updates, and artifacts are all produced by a peer you do not control. Treat them as untrusted input, verify results objectively (notebook 03), and keep human gates at consequential boundaries (notebook 05).